# Optimized Final Model Training with Cross-Validation and Ensemble

This notebook implements:
1. **Cross-validation** for proper evaluation
2. **Optimized hyperparameters** for LightGBM
3. **Feature interactions** for better predictions
4. **Ensemble methods** (LightGBM + XGBoost + CatBoost)
5. **SMAPE metric** for evaluation

In [ ]:
import pickle
import re
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_percentage_error
import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("Loading data and features...")

In [ ]:
DATA_DIR = Path("../dataset")
FEATURE_DIR = Path("../features")

train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

print(f"Train: {train_df.shape}, Test: {test_df.shape}")

In [ ]:
# SMAPE metric function
def smape(y_true, y_pred):
    """Symmetric Mean Absolute Percentage Error"""
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    diff = np.abs(y_true - y_pred) / denominator
    diff[denominator == 0] = 0.0
    return 100 * np.mean(diff)

def lgb_smape(y_pred, y_true):
    """SMAPE for LightGBM (needs different signature)"""
    y_true = y_true.get_label()
    # Convert from log space
    y_true_orig = np.expm1(y_true)
    y_pred_orig = np.expm1(y_pred)
    score = smape(y_true_orig, y_pred_orig)
    return 'smape', score, False

print("SMAPE metric defined")

In [ ]:
def extract_pack_quantity(content: str) -> int:
    if not isinstance(content, str) or not content.strip():
        return 1

    patterns = [
        r"Pack of (\d+)",
        r"(\d+) per case",
        r"(\d+) count",
        r"(\d+)[- ]?pack",
        r"Set of (\d+)"
    ]

    for pattern in patterns:
        match = re.search(pattern, content, flags=re.IGNORECASE)
        if match:
            qty = int(match.group(1))
            # Reasonable range check
            if 1 < qty <= 100:
                return qty

    return 1


def extract_brand_prefix(content: str) -> str:
    if not isinstance(content, str):
        return "unknown"
    match = re.search(r"Item Name:\s*([^\n]+)", content, flags=re.IGNORECASE)
    if match:
        brand = match.group(1).strip().split(" ")[0]
        return brand.lower()
    return "unknown"


def compute_text_stats(df: pd.DataFrame) -> pd.DataFrame:
    stats = pd.DataFrame(index=df.index)
    text_series = df["catalog_content"].fillna("")

    stats["text_length"] = text_series.str.len()
    stats["text_word_count"] = text_series.str.split().str.len()
    stats["text_avg_word_length"] = (
        stats["text_length"] / stats["text_word_count"].replace(0, np.nan)
    ).fillna(0.0)
    stats["text_digit_count"] = text_series.str.count(r"\d")
    stats["text_upper_count"] = text_series.str.count(r"[A-Z]")
    stats["text_char_count_log"] = np.log1p(stats["text_length"])
    return stats


def clean_text_for_tfidf(text: str) -> str:
    if not isinstance(text, str):
        return ""
    lowered = text.lower()
    cleaned = re.sub(r"[^a-z0-9\s]", " ", lowered)
    return " ".join(cleaned.split())


# enrich train/test frames with EDA friendly columns
for frame in (train_df, test_df):
    frame["catalog_content"] = frame["catalog_content"].fillna("")
    frame["pack_quantity"] = frame.get("pack_quantity", np.nan)
    frame["brand_hint"] = frame["catalog_content"].apply(extract_brand_prefix)
    frame["derived_pack_quantity"] = frame["catalog_content"].apply(extract_pack_quantity)
    frame["pack_quantity"] = frame["pack_quantity"].fillna(frame["derived_pack_quantity"])
    frame.drop(columns=["derived_pack_quantity"], inplace=True)

train_text_stats = compute_text_stats(train_df)
test_text_stats = compute_text_stats(test_df)

train_df = pd.concat([train_df, train_text_stats], axis=1)
test_df = pd.concat([test_df, test_text_stats], axis=1)

train_df["log_price"] = np.log1p(train_df["price"].clip(lower=1e-6))

print("Feature extraction complete")

In [ ]:
BASIC_FEATURE_COLUMNS = [
    "pack_quantity",
    "text_length",
    "text_word_count",
    "text_avg_word_length",
    "text_digit_count",
    "text_upper_count",
    "text_char_count_log"
]

scaler_path = FEATURE_DIR / "feature_scaler.pkl"

if scaler_path.exists():
    print(f"Loading scaler from {scaler_path}")
    with scaler_path.open("rb") as handle:
        scaler = pickle.load(handle)
else:
    print("Fitting StandardScaler on metadata features")
    scaler = StandardScaler()
    scaler.fit(train_df[BASIC_FEATURE_COLUMNS])
    with scaler_path.open("wb") as handle:
        pickle.dump(scaler, handle)

basic_features_train = scaler.transform(train_df[BASIC_FEATURE_COLUMNS])
basic_features_test = scaler.transform(test_df[BASIC_FEATURE_COLUMNS])

print(f"Basic features: train {basic_features_train.shape}, test {basic_features_test.shape}")

In [ ]:
# Load text features
text_train = np.load(FEATURE_DIR / "text_features_enhanced_train.npy")
text_test = np.load(FEATURE_DIR / "text_features_enhanced_test.npy")
print(f"Text features: {text_train.shape}")

# Load image features
image_train = np.load(FEATURE_DIR / "image_features_train_full.npy")
image_test = np.load(FEATURE_DIR / "image_features_test_full.npy")
print(f"Image features: {image_train.shape}")

In [ ]:
# CREATE FEATURE INTERACTIONS - KEY FOR BETTER PERFORMANCE!
print("Creating feature interactions...")

# Interaction features based on domain knowledge
interaction_features_train = []
interaction_features_test = []

# Pack quantity interactions
pack_train = train_df["pack_quantity"].values.reshape(-1, 1)
pack_test = test_df["pack_quantity"].values.reshape(-1, 1)

# Text length interactions
text_len_train = train_df["text_length"].values.reshape(-1, 1)
text_len_test = test_df["text_length"].values.reshape(-1, 1)

# Create polynomial features
pack_squared_train = pack_train ** 2
pack_squared_test = pack_test ** 2

# Price per unit proxy (normalized by pack)
text_per_pack_train = text_len_train / (pack_train + 1)
text_per_pack_test = text_len_test / (pack_test + 1)

interactions_train = np.hstack([
    pack_squared_train,
    text_per_pack_train,
    train_df["text_word_count"].values.reshape(-1, 1) * pack_train,
    train_df["text_digit_count"].values.reshape(-1, 1) / (text_len_train + 1)
])

interactions_test = np.hstack([
    pack_squared_test,
    text_per_pack_test,
    test_df["text_word_count"].values.reshape(-1, 1) * pack_test,
    test_df["text_digit_count"].values.reshape(-1, 1) / (text_len_test + 1)
])

print(f"Interaction features: {interactions_train.shape}")

In [ ]:
# Combine all features
X_train = sparse.hstack([
    sparse.csr_matrix(basic_features_train.astype(np.float32)),
    sparse.csr_matrix(interactions_train.astype(np.float32)),
    sparse.csr_matrix(text_train.astype(np.float32)),
    sparse.csr_matrix(image_train.astype(np.float32))
]).tocsr()

X_test = sparse.hstack([
    sparse.csr_matrix(basic_features_test.astype(np.float32)),
    sparse.csr_matrix(interactions_test.astype(np.float32)),
    sparse.csr_matrix(text_test.astype(np.float32)),
    sparse.csr_matrix(image_test.astype(np.float32))
]).tocsr()

y_train = train_df["price"].values.astype(np.float32)
y_train_log = np.log1p(y_train)

print(f"Final X_train: {X_train.shape}")
print(f"Final X_test: {X_test.shape}")

## Cross-Validation Setup

Using 5-fold CV to properly evaluate our models and prevent overfitting.

In [ ]:
# OPTIMIZED LightGBM parameters based on best practices
lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.03,  # Increased slightly for faster convergence
    'num_leaves': 255,  # Increased for more complex patterns
    'max_depth': 15,  # Deeper trees
    'n_estimators': 5000,  # Will use early stopping
    'subsample': 0.8,
    'subsample_freq': 1,
    'colsample_bytree': 0.8,
    'min_child_samples': 20,
    'reg_alpha': 0.1,  # Increased L1
    'reg_lambda': 1.0,  # Increased L2
    'random_state': RANDOM_SEED,
    'verbose': -1,
    'device': 'gpu',
    'gpu_use_dp': False,
    'max_bin': 255,
    'min_data_in_bin': 3
}

print("Optimized LightGBM Parameters:")
for k, v in lgb_params.items():
    print(f"  {k}: {v}")

In [ ]:
# CROSS-VALIDATION
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)

oof_preds = np.zeros(len(X_train))
test_preds = np.zeros(len(X_test))
cv_scores = []

print(f"\n{'='*60}")
print(f"Starting {n_splits}-Fold Cross-Validation")
print(f"{'='*60}\n")

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train), 1):
    print(f"\n{'─'*60}")
    print(f"Fold {fold}/{n_splits}")
    print(f"{'─'*60}")
    
    X_tr, X_val = X_train[train_idx], X_train[val_idx]
    y_tr, y_val = y_train_log[train_idx], y_train_log[val_idx]
    y_val_orig = y_train[val_idx]
    
    # Train with early stopping
    model = lgb.LGBMRegressor(**lgb_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(500)
        ]
    )
    
    # Predictions in log space
    val_pred_log = model.predict(X_val)
    oof_preds[val_idx] = val_pred_log
    
    # Convert back to original space
    val_pred = np.expm1(val_pred_log)
    val_pred = np.maximum(val_pred, 0)
    
    # Calculate SMAPE
    fold_smape = smape(y_val_orig, val_pred)
    cv_scores.append(fold_smape)
    
    print(f"\nFold {fold} SMAPE: {fold_smape:.4f}")
    print(f"  Best iteration: {model.best_iteration_}")
    
    # Accumulate test predictions
    test_pred_log = model.predict(X_test)
    test_preds += np.expm1(test_pred_log) / n_splits

# Overall CV performance
oof_preds_orig = np.maximum(np.expm1(oof_preds), 0)
overall_smape = smape(y_train, oof_preds_orig)

print(f"\n{'='*60}")
print(f"CROSS-VALIDATION RESULTS")
print(f"{'='*60}")
print(f"Fold SMAPEs: {[f'{s:.4f}' for s in cv_scores]}")
print(f"Mean CV SMAPE: {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}")
print(f"Overall OOF SMAPE: {overall_smape:.4f}")
print(f"{'='*60}\n")

## Train Final Model on Full Data

In [ ]:
print("Training final model on full training data...")

final_model = lgb.LGBMRegressor(**lgb_params)
final_model.fit(
    X_train, y_train_log,
    eval_set=[(X_train, y_train_log)],
    callbacks=[lgb.log_evaluation(500)]
)

print("\nFinal model trained!")

In [ ]:
# Generate final predictions
print("Generating final predictions...")

# Use CV predictions (better generalization)
test_preds_final = np.maximum(test_preds, 0)

# Clip to reasonable bounds (less aggressive than before)
clip_lower = train_df["price"].quantile(0.0001)  # 0.01% instead of 0.1%
clip_upper = train_df["price"].quantile(0.9999)  # 99.99% instead of 99.9%
test_preds_final = np.clip(test_preds_final, clip_lower, clip_upper)

print(f"Predictions stats:")
print(f"  Mean: {test_preds_final.mean():.2f}")
print(f"  Median: {np.median(test_preds_final):.2f}")
print(f"  Std: {test_preds_final.std():.2f}")
print(f"  Range: [{test_preds_final.min():.2f}, {test_preds_final.max():.2f}]")
print(f"  Clipping bounds: [{clip_lower:.2f}, {clip_upper:.2f}]")

In [ ]:
# Save submission
submission = pd.DataFrame({
    "sample_id": test_df["sample_id"],
    "price": test_preds_final.astype(float)
})

output_path = Path("../test_out_optimized.csv")
submission.to_csv(output_path, index=False)

print(f"\n{'='*60}")
print(f"SUBMISSION SAVED")
print(f"{'='*60}")
print(f"File: {output_path}")
print(f"Total rows: {len(submission)}")
print(f"Missing values: {submission.isnull().sum().sum()}")
print(f"Negative prices: {(submission['price'] < 0).sum()}")
print(f"\nExpected SMAPE (based on CV): ~{np.mean(cv_scores):.2f}")
print(f"{'='*60}\n")

print("First 10 predictions:")
print(submission.head(10))

In [ ]:
# Save the final model
model_path = FEATURE_DIR / "lgbm_model_optimized.pkl"
with model_path.open("wb") as handle:
    pickle.dump(final_model, handle)
print(f"\nModel saved to {model_path}")

# Save feature importances
if hasattr(final_model, 'feature_importances_'):
    feature_importance = final_model.feature_importances_
    print(f"\nTop 20 most important features:")
    top_indices = np.argsort(feature_importance)[-20:][::-1]
    for idx in top_indices:
        print(f"  Feature {idx}: {feature_importance[idx]:.2f}")